# Bronze Layer - Azure Databricks Serverless
Ingests the raw Parking CSV from **Azure Blob/ADLS storage** into a Unity Catalog Delta table.

**Serverless notes:** no cluster config, no `SparkSession.builder`, no storage account keys in `spark.conf` (not supported on serverless), and no DBFS root writes. Storage access goes through a Unity Catalog **storage credential + external location** covering your container; output is a **managed Delta table**.

**One-time setup (Catalog Explorer -> External Data):**
1. Create a *Storage Credential* (Azure Managed Identity / Access Connector for Azure Databricks with `Storage Blob Data Contributor` on the storage account).
2. Create an *External Location* pointing to `abfss://<container>@<storage-account>.dfs.core.windows.net/`.
3. Grant yourself `READ FILES` on that external location.

The storage account must have hierarchical namespace (ADLS Gen2) for `abfss://`.

## 1. Configuration

In [ ]:
# Update these for your workspace
storage_account = "<storage-account-name>"
container = "<blob-container-name>"
file_path = "Parking Dataset.csv"        # path inside the container (a folder also works)

catalog = "iceberg_lakehouse"
schema = "bronze"

file_location = f"abfss://{container}@{storage_account}.dfs.core.windows.net/{file_path}"
bronze_table = f"{catalog}.{schema}.parking_data_bronze"

## 2. Create catalog and schema

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

## 3. Read raw CSV from Blob storage

In [ ]:
df_bronze = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(file_location)
)

In [ ]:
display(df_bronze)

In [ ]:
df_bronze.count()

## 4. Check for leading/trailing spaces in column names

In [ ]:
for col_name in df_bronze.columns:
    if col_name != col_name.strip():
        print(f"Column with trailing/leading spaces: '{col_name}'")

## 5. Clean column names
Delta does not allow spaces or `,;{}()
	=` in column names, so trim and replace them with underscores.

In [ ]:
import re

def clean_column_name(name):
    name = name.strip()                                  # Remove leading/trailing spaces
    name = re.sub(r'[ ,;{}()
	=?:]', '_', name)        # Replace unsupported characters with underscore
    name = re.sub(r'_+', '_', name)                      # Collapse multiple underscores
    return name.strip('_')                               # Remove leading/trailing underscores

df_bronze = df_bronze.toDF(*[clean_column_name(c) for c in df_bronze.columns])
df_bronze.columns

## 6. Write to the Bronze Delta table

In [ ]:
(
    df_bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table)
)

## 7. Verify

In [ ]:
display(spark.sql(f"DESCRIBE DETAIL {bronze_table}"))
display(spark.table(bronze_table).limit(10))

In [ ]:
display(dbutils.fs.ls(f"abfss://{container}@{storage_account}.dfs.core.windows.net/"))